# 01 — HAR-RV (weekly silver volatility)

First two models on the volatility ladder:

- **Naïve** — $\hat{\text{RV}}_t = \text{RV}_{t-1}$. The floor. Volatility has a strong
  AR(1) component, so last week's RV is already a hard baseline to beat.
- **HAR-RV** (Corsi 2009) — OLS regression of RV on its short / medium / long trailing
  averages.

Features come from `volatility_weekly.csv`, built in `00_features.ipynb` — run that
notebook first.


## Setup


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from vol_utils import vol_evaluate, vol_period_metrics, vol_diebold_mariano
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')

train_df = frame[frame['split'] == 'train']
val_df   = frame[frame['split'] == 'val']
test_df  = frame[frame['split'] == 'test']
trval_df = frame[frame['split'] != 'test']     # HAR-RV is plain OLS -> train+val fitted as one

FEATS_HAR = ['rv_w_lag1', 'rv_m_lag1', 'rv_q_lag1']
print(f'train+val: {len(trval_df)}   test: {len(test_df)}')

train+val: 405   test: 175


## 1. Naïve baseline — $\hat{\text{RV}}_t = \text{RV}_{t-1}$

The naïve prediction is simply last week's RV, which is exactly the `rv_w_lag1` column.
Because volatility is highly persistent this is a genuinely strong baseline — every
other model has to beat it to justify itself.

(Its DCA is ≈ 0 by construction: predicting last week's RV implies *no* change, so the
naïve model can never call a direction. It is a floor for RMSE/MAE, not for DCA.)


In [2]:
y_test    = test_df['target'].values
prev_test = test_df['rv_w_lag1'].values        # RV_{t-1}

results = []
results.append(vol_evaluate('Naive (RV_t-1)', y_test, prev_test, prev_test))


Naive (RV_t-1)                  RMSE=0.03979  MAE=0.02013  R2=-0.095  DCA=0.000


## 2. HAR-RV (Corsi 2009)

**HAR-RV** stands for the *Heterogeneous Autoregressive model of Realised Volatility*.
It is a simple linear regression that predicts realised volatility from its own past
values measured over several horizons. The "heterogeneous" in the name refers to the
economic story behind it: different market participants operate on different time
horizons — short-term traders care about last week, medium-term traders about last
month, longer-term traders about last quarter — so the volatility observed today
reflects a mixture of all of those horizons.

Concretely it is OLS on the three HAR features built in `00_features.ipynb`:

$$\text{RV}_t = \beta_0 + \beta_w \text{RV}^{(w)}_{t-1} + \beta_m \text{RV}^{(m)}_{t-1} + \beta_q \text{RV}^{(q)}_{t-1} + \varepsilon_t$$

Despite its simplicity HAR-RV is the standard volatility benchmark in the literature,
because those three lags capture the long-memory persistence of volatility well. Train
and val are pooled for the final fit since OLS has no hyperparameters to tune.


In [3]:
X_tr = sm.add_constant(trval_df[FEATS_HAR])
y_tr = trval_df['target']
X_te = sm.add_constant(test_df[FEATS_HAR])

har_fit = sm.OLS(y_tr, X_tr).fit()
print(har_fit.summary().tables[1])

har_pred = har_fit.predict(X_te).values
results.append(vol_evaluate('HAR-RV', y_test, har_pred, prev_test))


                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.003      3.196      0.002       0.003       0.014
rv_w_lag1      0.0668      0.062      1.074      0.284      -0.056       0.189
rv_m_lag1      0.3971      0.126      3.155      0.002       0.150       0.644
rv_q_lag1      0.2799      0.129      2.173      0.030       0.027       0.533
HAR-RV                          RMSE=0.03153  MAE=0.01576  R2=+0.313  DCA=0.714


## 3. Sub-period breakdown

RMSE and DCA for HAR-RV split by calendar year, using the shared `PERIODS` definition
from `eval_utils`. This shows whether the model holds up across the choppy 2023, the
2024 bull start, the 2025 bull run, and 2026 YTD.


In [4]:
period_har = vol_period_metrics(y_test, har_pred, prev_test, test_df.index, PERIODS)
period_har.to_csv('../../data/processed/period_har_volatility.csv')
period_har.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0148,0.0122,0.6923
2024 (bull start),52,0.0145,0.0114,0.7885
2025 (bull run),52,0.0222,0.0155,0.6923
2026 (YTD),19,0.0815,0.0381,0.6316
── Full test ──,175,0.0315,0.0158,0.7143


## 4. Sentiment ablation — does Reddit sentiment add to HAR-RV?

HAR-RV uses only the silver RV's own past. This ablation asks whether **public
sentiment** carries information about next week's volatility *beyond* what the HAR lags
already capture — the news→volatility channel that motivated the sentiment features in
`00_features` §5.

Five rungs, every one a plain OLS, all fitted and scored on the **same sample** (the
weeks where the Reddit features exist — only the two boundary weeks are dropped), so
the comparison is fully attributable:

| Rung | Features |
|---|---|
| `HAR` | 3 HAR lags — the baseline |
| `HAR+Attention` | + `reddit_attention_lag1` (log post volume) |
| `HAR+SentIntensity` | + `reddit_sent_abs_lag1`, `reddit_sent_disp_lag1` |
| `HAR+Attention+SentIntensity` | + all three sentiment features |
| `HAR+VIX` | + `vix_rv_lag1` — **the control** |

**Why the `HAR+VIX` control.** HAR-RV is univariate, so *any* extra regressor
correlated with market stress will lift the fit a little, and VIX realised vol is
already a fear/volatility gauge. If a sentiment rung beats `HAR` but not `HAR+VIX`, the
"sentiment effect" is really just a VIX proxy; only beating `HAR+VIX` too is a genuine
incremental result. Significance is read from **QLIKE-DM** vs the `HAR` baseline —
squared-error DM is near-powerless on this heavy-tailed target (see `evaluation.ipynb`).

In [5]:
FEATS_SENT_ATTN = ['reddit_attention_lag1']
FEATS_SENT_INT  = ['reddit_sent_abs_lag1', 'reddit_sent_disp_lag1']

LADDER = {
    'HAR':                          FEATS_HAR,
    'HAR+Attention':                FEATS_HAR + FEATS_SENT_ATTN,
    'HAR+SentIntensity':            FEATS_HAR + FEATS_SENT_INT,
    'HAR+Attention+SentIntensity':  FEATS_HAR + FEATS_SENT_ATTN + FEATS_SENT_INT,
    'HAR+VIX':                      FEATS_HAR + ['vix_rv_lag1'],
}

# common sample: every rung fitted/scored on the weeks where sentiment exists
sent_cols = FEATS_SENT_ATTN + FEATS_SENT_INT
abl       = frame.dropna(subset=sent_cols)
abl_trval = abl[abl['split'] != 'test']
abl_test  = abl[abl['split'] == 'test']
y_abl, prev_abl = abl_test['target'].values, abl_test['rv_w_lag1'].values
print(f'ablation sample: train+val={len(abl_trval)}  test={len(abl_test)}  '
      f'(headline HAR used {len(trval_df)}/{len(test_df)})\n')

abl_pred, abl_results = {}, []
for name, feats in LADDER.items():
    fit  = sm.OLS(abl_trval['target'], sm.add_constant(abl_trval[feats])).fit()
    pred = fit.predict(sm.add_constant(abl_test[feats])).values
    abl_pred[name] = pred
    abl_results.append(vol_evaluate(name, y_abl, pred, prev_abl))

# QLIKE-DM: each rung vs the HAR baseline (negative DM = rung better)
print()
dm = {}
for name in LADDER:
    if name == 'HAR':
        continue
    dm[name] = vol_diebold_mariano(y_abl, abl_pred[name], abl_pred['HAR'],
                                   name, 'HAR', loss='qlike')
# does the full sentiment rung beat the VIX control, not just bare HAR?
print()
vol_diebold_mariano(y_abl, abl_pred['HAR+Attention+SentIntensity'], abl_pred['HAR+VIX'],
                    'HAR+Att+Int', 'HAR+VIX', loss='qlike')

abl_df = pd.DataFrame(abl_results)
abl_df['dm_qlike']   = abl_df['model'].map(lambda m: dm[m]['dm'] if m in dm else np.nan)
abl_df['dm_qlike_p'] = abl_df['model'].map(lambda m: dm[m]['p']  if m in dm else np.nan)
abl_df.to_csv('../../data/processed/metrics_har_sentiment_volatility.csv', index=False)
print('\nSaved metrics_har_sentiment_volatility.csv')
abl_df.round(5)

ablation sample: train+val=404  test=174  (headline HAR used 405/175)

HAR                             RMSE=0.03151  MAE=0.01564  R2=+0.316  DCA=0.718
HAR+Attention                   RMSE=0.03143  MAE=0.01561  R2=+0.320  DCA=0.713
HAR+SentIntensity               RMSE=0.03146  MAE=0.01569  R2=+0.318  DCA=0.713
HAR+Attention+SentIntensity     RMSE=0.03146  MAE=0.01579  R2=+0.319  DCA=0.713
HAR+VIX                         RMSE=0.03159  MAE=0.01608  R2=+0.313  DCA=0.684

HAR+Attention vs HAR           [qlike]  DM=-2.833  p=0.005  **
HAR+SentIntensity vs HAR           [qlike]  DM=-2.857  p=0.004  **
HAR+Attention+SentIntensity vs HAR           [qlike]  DM=-1.831  p=0.067  (ns)
HAR+VIX      vs HAR           [qlike]  DM=+1.030  p=0.303  (ns)

HAR+Att+Int  vs HAR+VIX       [qlike]  DM=-1.825  p=0.068  (ns)

Saved metrics_har_sentiment_volatility.csv


,model,rmse,mae,r2,dca,dm_qlike,dm_qlike_p
0,HAR,0.03151,0.01564,0.31615,0.71839,NaN,NaN
1,HAR+Attention,0.03143,0.01561,0.31976,0.71264,-2.83258,0.00462
2,HAR+SentIntensity,0.03146,0.01569,0.31827,0.71264,-2.85690,0.00428
3,HAR+Attention+SentIntensity,0.03146,0.01579,0.31868,0.71264,-1.83097,0.06711
4,HAR+VIX,0.03159,0.01608,0.31286,0.68391,1.03002,0.30300


**Reading the table.** Compare each sentiment rung's RMSE against the `HAR` row;
`dm_qlike` / `dm_qlike_p` give the QLIKE-DM stat and p-value of that rung vs `HAR`
(negative = the rung is better; the `HAR` row is blank). Two questions:

1. **Does sentiment beat bare HAR?** A negative, significant `dm_qlike` on
   `HAR+Attention`, `HAR+SentIntensity` or the combined rung says yes.
2. **Does it survive the VIX control?** Compare the sentiment rungs' RMSE to `HAR+VIX`,
   and read the explicit `HAR+Att+Int vs HAR+VIX` DM line printed above. If the
   sentiment rungs only match `HAR+VIX`, the signal is just a market-vol proxy; if they
   beat it, Reddit sentiment carries genuinely incremental volatility information.

A null result here is still thesis-relevant: *"sentiment adds nothing once the HAR
lags — and VIX — are in"* is clean semi-strong-form evidence, consistent with the
returns side of the thesis.

## 5. Save outputs

- `metrics_har_volatility.csv` — Naïve + HAR-RV headline metrics
- `pred_har_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`
- `metrics_har_sentiment_volatility.csv` — the §4 sentiment-ablation table (saved above)

In [6]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_har_volatility.csv', index=False)

pred_har = pd.DataFrame({'actual': y_test, 'prev': prev_test,
                         'naive': prev_test, 'har': har_pred}, index=test_df.index)
pred_har.to_csv('../../data/processed/pred_har_volatility.csv', index_label='Date')
print('Saved metrics_har_volatility.csv + pred_har_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_har_volatility.csv + pred_har_volatility.csv


,model,rmse,mae,r2,dca
0,Naive (RV_t-1),0.03979,0.02013,-0.09462,0.00000
1,HAR-RV,0.03153,0.01576,0.31255,0.71429
